# **Calculate Course Similarity using BoW Features**


Similarity measurement between items is the foundation of many recommendation algorithms, especially for content-based recommendation systems. For example, if a new course is similar to a user's enrolled courses, we could recommend that new similar course to the user. Alternatively, if user A is similar to user B, we can recommend some of user B's courses to user A (the unseen courses) because they may have similar interests.


Previously, I extracted Bag of Words (BoW) features from course textual content. Given these course BoW feature vectors, I can now apply similarity measurements to calculate course similarity.


## Objectives


In this analysis, I will:

* Calculate the similarity between any two courses using BoW feature vectors
* Identify courses similar to a target course based on content similarity


----


## Environment Setup


In [1]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import gensim
import pandas as pd
import nltk as nltk

from scipy.spatial.distance import cosine
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk import ngrams
from gensim import corpora

%matplotlib inline

In [2]:
# Setting random state for reproducibility
rs = 123

## Calculating Cosine Similarity Between Two Example Courses


Let's start with two simple example courses:


In [3]:
course1 = "machine learning for everyone"
course2 = "machine learning for beginners"

First, I'll tokenize them using the split() method:


In [4]:
tokens = set(course1.split() + course2.split())
tokens = list(tokens)
tokens

['machine', 'beginners', 'for', 'everyone', 'learning']

Next, I'll generate BoW features (token counts) for these two courses:

In [5]:
def generate_sparse_bow(course, vocab_set):
    """
    Generate a sparse bag-of-words (BoW) representation for a given course.

    Parameters:
    course (str): The input course text to generate the BoW representation for.
    vocab_set (list): The vocabulary set containing all possible tokens.

    Returns:
    list: A sparse BoW representation where each element corresponds to the presence (1) or absence (0)
    of a word in the input course text.
    """

    # Initialize an empty list to store the BoW vector
    bow_vector = []

    # Tokenize the course text by splitting it into words
    words = course.split()

    # Iterate through all unique words (tokens) in the vocabulary set
    for token in vocab_set:
        # Check if the token is present in the course text
        if token in words:
            # If the token is present, append 1 to the BoW vector
            bow_vector.append(1)
        else:
            # If the token is not present, append 0 to the BoW vector
            bow_vector.append(0)

    # Return the sparse BoW vector
    return bow_vector

In [6]:
bow1 = generate_sparse_bow(course1, tokens)
bow1

[1, 0, 1, 1, 1]

In [7]:
bow2 = generate_sparse_bow(course2, tokens)
bow2

[1, 1, 1, 0, 1]

From the outputs above, we can see the two vectors are very similar. Only two dimensions are different.

Now I'll apply the cosine similarity measurement to the two vectors:

In [8]:
cos_sim = 1 - cosine(bow1, bow2)
print(f"The cosine similarity between course `{course1}` and course `{course2}` is {round(cos_sim, 2) * 100}%")

The cosine similarity between course `machine learning for everyone` and course `machine learning for beginners` is 75.0%


### Exploring Other Similarity Measures

In [9]:
# Calculate Euclidean distance
euclidean_distance = np.sqrt(np.sum((np.array(bow1) - np.array(bow2))**2))
print("The Euclidean distance = " + str(euclidean_distance))

# Calculate Jaccard similarity
intersection = set()
for word in set(course1.split()):
    if word in set(course2.split()):
        intersection.add(word)
jaccard_similarity = len(intersection)/len(tokens)
print("The Jaccard similarity = " + str(jaccard_similarity))

The Euclidean distance = 1.4142135623730951
The Jaccard similarity = 0.6


## Finding Similar Courses to "Machine Learning with Python"


Now I'll work with real course BoW feature vectors to find courses similar to "Machine Learning with Python".

In [11]:
# Load the BoW features as Pandas dataframe
bows_df = pd.read_csv("data/courses_bows.csv")
bows_df = bows_df[['doc_id', 'token', 'bow']]

In [12]:
bows_df.head(10)

,doc_id,token,bow
0,ML0201EN,ai,2
1,ML0201EN,apps,2
2,ML0201EN,build,2
3,ML0201EN,cloud,1
4,ML0201EN,coming,1
5,ML0201EN,create,1
6,ML0201EN,data,1
7,ML0201EN,developer,1
8,ML0201EN,found,1
9,ML0201EN,fun,1


The `bows_df` dataframe contains the BoW feature vectors for each course in a vertical and dense format. It has three columns: `doc_id` (course id), `token` (token value), and `bow` (token count).

Now I'll load the course content dataset containing titles and descriptions:

In [14]:
# Load the course dataframe
course_df = pd.read_csv("data/course_processed.csv")

In [15]:
course_df.head(10)

,COURSE_ID,TITLE,DESCRIPTION
0,ML0201EN,robots are coming build iot apps with watson ...,have fun with iot and learn along the way if ...
1,ML0122EN,accelerating deep learning with gpu,training complex deep learning models with lar...
2,GPXX0ZG0EN,consuming restful services using the reactive ...,learn how to use a reactive jax rs client to a...
3,RP0105EN,analyzing big data in r using apache spark,apache spark is a popular cluster computing fr...
4,GPXX0Z2PEN,containerizing packaging and running a sprin...,learn how to containerize package and run a ...
5,CNSC02EN,cloud native security conference data security,introduction to data security on cloud
6,DX0106EN,data science bootcamp with r for university pr...,a multi day intensive in person data science ...
7,GPXX0FTCEN,learn how to use docker containers for iterati...,learn how to use docker containers for iterati...
8,RAVSCTEST1,scorm test 1,scron test course
9,GPXX06RFEN,create your first mongodb database,in this guided project you will get started w...


Let's find the title and description for the course ID `ML0101ENv3`:

In [16]:
course_df[course_df['COURSE_ID'] == 'ML0101ENv3']

,COURSE_ID,TITLE,DESCRIPTION
158,ML0101ENv3,machine learning with python,machine learning can be an incredibly benefici...


We can see it's a "Machine Learning with Python" course, so we can expect other machine learning or Python-related courses to be similar.

Let's examine its associated BoW features:

In [17]:
ml_course = bows_df[bows_df['doc_id'] == 'ML0101ENv3']
ml_course

,doc_id,token,bow
2747,ML0101ENv3,course,1
2748,ML0101ENv3,learning,4
2749,ML0101ENv3,machine,3
2750,ML0101ENv3,need,1
2751,ML0101ENv3,get,1
2752,ML0101ENv3,started,1
2753,ML0101ENv3,python,2
2754,ML0101ENv3,tool,1
2755,ML0101ENv3,tools,1
2756,ML0101ENv3,predict,1


The BoW feature vector is in vertical format, but feature vectors are typically used in horizontal format. I can transpose the feature vector using the Pandas `pivot()` method:

In [18]:
ml_courseT = ml_course.pivot(index=['doc_id'], columns='token').reset_index(level=[0])
ml_courseT

doc_id        bow                                              \
token             beneficial course free future get give hidden insights   
0      ML0101ENv3          1      1    1      1   1    1      1        1   

                                                                          \
token learning machine need predict python started supervised tool tools   
0            4       3    1       1      2       1          1    1     1   

                           
token trends unsupervised  
0          1            1

To compare the BoWs of any two courses (which may have different tokens), I need to create a union token set and transpose them. I'll use the provided `pivot_two_bows` method:

In [19]:
def pivot_two_bows(basedoc, comparedoc):
    """
    Pivot two bag-of-words (BoW) representations for comparison.

    Parameters:
    basedoc (DataFrame): DataFrame containing the bag-of-words representation for the base document.
    comparedoc (DataFrame): DataFrame containing the bag-of-words representation for the document to compare.

    Returns:
    DataFrame: A DataFrame with pivoted BoW representations for the base and compared documents,
    facilitating direct comparison of word occurrences between the two documents.
    """

    # Create copies of the input DataFrames to avoid modifying the originals
    base = basedoc.copy()
    base['type'] = 'base'  # Add a 'type' column indicating base document
    compare = comparedoc.copy()
    compare['type'] = 'compare'  # Add a 'type' column indicating compared document

    # Concatenate the two DataFrames vertically
    join = pd.concat([base, compare])

    # Pivot the concatenated DataFrame based on 'doc_id' and 'type', with words as columns
    joinT = join.pivot(index=['doc_id', 'type'], columns='token').fillna(0).reset_index(level=[0, 1])

    # Assign meaningful column names to the pivoted DataFrame
    joinT.columns = ['doc_id', 'type'] + [t[1] for t in joinT.columns][2:]

    # Return the pivoted DataFrame for comparison
    return joinT

In [20]:
course1 = bows_df[bows_df['doc_id'] == 'ML0151EN']
course2 = bows_df[bows_df['doc_id'] == 'ML0101ENv3']

bow_vectors = pivot_two_bows(course1, course2)
bow_vectors

,doc_id,type,approachable,basics,beneficial,comparison,course,dives,free,future,...,relates,started,statistical,supervised,tool,tools,trends,unsupervised,using,vs
0,ML0101ENv3,compare,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,...,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0
1,ML0151EN,base,1.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,...,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0


Now I can calculate their similarity using the cosine method:

In [21]:
similarity = 1 - cosine(bow_vectors.iloc[0, 2:], bow_vectors.iloc[1, 2:])
similarity

0.662622139954909

### Finding All Courses Similar to "Machine Learning with Python"

Now I'll find all courses similar to "Machine Learning with Python" (ML0101ENv3) and show their titles and descriptions.

In [22]:
course_df[course_df['COURSE_ID'] == 'ML0101ENv3']

,COURSE_ID,TITLE,DESCRIPTION
158,ML0101ENv3,machine learning with python,machine learning can be an incredibly benefici...


I'll set a similarity threshold of 0.5 to determine if two courses are similar enough.

In [23]:
base_course = bows_df[bows_df["doc_id"] == "ML0101ENv3"]
unique_ids = course_df["COURSE_ID"].to_list()

similar_courses_df = pd.DataFrame({
    "course_id": [],
    "title": [],
    "description": []
})

for unique_id in unique_ids:
    compare_course = bows_df[bows_df["doc_id"] == unique_id]
    vecs = pivot_two_bows(base_course, compare_course)
    sim = 1 - cosine(vecs.iloc[0, 2:], vecs.iloc[1, 2:])
    if (sim > 0.5) & (unique_id != vecs.iloc[0, 0]):
        new_row = pd.DataFrame({
            "course_id": [unique_id],
            "title": [course_df[course_df["COURSE_ID"] == unique_id]["TITLE"]],
            "description": [course_df[course_df["COURSE_ID"] == unique_id]["DESCRIPTION"]]
        })
        similar_courses_df = pd.concat([similar_courses_df, new_row], ignore_index=True)

similar_courses_df

,course_id,title,description
0,ML0109EN,157 machine learning dimensionality reduc...,157 machine learning dimensionality reduc...
1,ML0151EN,"200 machine learning with r Name: TITLE, dt...",200 this machine learning with r course div...
2,excourse46,"259 machine learning Name: TITLE, dtype: ob...",259 machine learning is the science of gett...
3,excourse47,"260 machine learning for all Name: TITLE, d...",260 machine learning often called artifici...
4,excourse60,273 introduction to tensorflow for artifici...,273 if you are a software developer who wan...


## Advanced Similarity Techniques

Beyond basic cosine similarity with BoW features, several advanced techniques could further enhance our recommendation system:

1. **TF-IDF weighting**: Instead of raw word counts, we could use TF-IDF to give higher weights to terms that are important to a specific document but not common across all documents.

2. **Word embeddings**: Models like Word2Vec, GloVe, or BERT embeddings can capture semantic relationships between words better than BoW features.

3. **Hybrid approaches**: Combining content-based similarity with collaborative filtering could provide more personalized and accurate recommendations.

4. **Topic modeling**: Using techniques like LDA (Latent Dirichlet Allocation) to identify topics in courses and compare them at a more abstract level.

## Conclusion

In this analysis, I've demonstrated how to calculate similarities between courses using their Bag of Words feature vectors. The cosine similarity measurement provides a straightforward and effective method for identifying related courses.

The approach I've implemented forms the foundation of a content-based recommendation system. By identifying courses with high similarity to a user's previously enrolled or liked courses, we can generate relevant recommendations that align with their interests and learning goals.

For a production recommendation system, this approach would be further enhanced with additional techniques, such as collaborative filtering, to provide more personalized and diverse recommendations.